# synthkit worked example: Wine Quality

1,599 rows, 12 almost-entirely-continuous columns, no nulls at all -- a clean stress test for
whether the Gaussian copula preserves the *entire* correlation matrix (66 pairs across 12
columns), not just one hand-picked pair.

Running this during development caught a severe bug: a low-cardinality numeric column (the
`quality` rating, 3-8) was ordered by frequency instead of by value for the copula, which
silently destroyed its correlation with every other column. Fixed; see `CHANGELOG.md`.

In [1]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

import synthkit as sk

DATA_DIR = Path("../data")
DATA_PATH = DATA_DIR / "winequality-red.csv"
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    subprocess.run(["curl", "-sL", "-o", str(DATA_PATH), DATA_URL], check=True)

df = pd.read_csv(DATA_PATH, sep=";")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [2]:
profile = sk.fit(df)
print(f"{len(df)} rows, {len(df.columns)} columns")
print(f"all copula-eligible: {len(profile.copula_columns) == len(df.columns)}")

1599 rows, 12 columns
all copula-eligible: True


## Full correlation matrix fidelity

In [3]:
synthetic = sk.emit(profile, n=len(df), seed=0)

real_corr = df.corr().to_numpy()
synth_corr = synthetic.astype(float).corr().to_numpy()

triu = np.triu_indices_from(real_corr, k=1)
abs_error = np.abs(real_corr[triu] - synth_corr[triu])

print(f"mean absolute error across all 66 pairs: {abs_error.mean():.4f}")
print(f"max absolute error: {abs_error.max():.4f}")
print(f"pairs within 0.05: {(abs_error < 0.05).sum()} / {len(abs_error)}")

mean absolute error across all 66 pairs: 0.0398
max absolute error: 0.2963
pairs within 0.05: 45 / 66


## `quality`'s correlation with every other column, individually

In [4]:
real_q = df.corr()["quality"]
synth_q = synthetic.astype(float).corr()["quality"]

for col in df.columns:
    if col == "quality":
        continue
    print(f"{col:22s} real={real_q[col]:+.3f} synth={synth_q[col]:+.3f}")

fixed acidity          real=+0.124 synth=+0.092
volatile acidity       real=-0.391 synth=-0.318
citric acid            real=+0.226 synth=+0.173
residual sugar         real=+0.014 synth=+0.081
chlorides              real=-0.129 synth=-0.142
free sulfur dioxide    real=-0.051 synth=-0.032
total sulfur dioxide   real=-0.185 synth=-0.114
density                real=-0.175 synth=-0.128
pH                     real=-0.058 synth=-0.060
sulphates              real=+0.251 synth=+0.305
alcohol                real=+0.476 synth=+0.383


## Privacy check

In [5]:
report = sk.check(synthetic, profile, real=df, min_dcr_ratio=0.5)
print(f"dcr_ratio: {report.dcr_ratio:.1f}")
print(f"exact_matches: {report.exact_matches}")
print(f"mean per-column KS statistic: {np.mean(list(report.ks_by_column.values())):.4f}")

dcr_ratio: 25859555.1
exact_matches: 0
mean per-column KS statistic: 0.0267
